# FabricSort: ระบบจำแนกเศษผ้าเพื่อนำไปรีไซเคิล

**Applied Machine Learning** · จำแนกรูปเศษผ้า 1 รูปเป็น **Cotton / Polyester / Silk / Viscose** พร้อมค่าความมั่นใจ

Notebook นี้ทำงานครบตั้งแต่โหลดข้อมูล เทรนและเปรียบเทียบ 3 โมเดล จนถึงเปิดแอป Streamlit กด **Run All** ได้เลย โดยแก้เฉพาะ cell **Config (ส่วนที่ 0)**

> **เวอร์ชันนี้:** เก็บ ResNet50 ไว้ และเปลี่ยนอีก 2 โมเดลเป็นรุ่นใหม่กว่า (ConvNeXt V2-Tiny, DINOv3 ViT-S/16) แทน SVM และ KNN เดิม
> notebook เวอร์ชันก่อนหน้า (SVM / KNN / ResNet50) เก็บไว้ที่ `fabric_classification_old.ipynb` และ `backup_original_2026-10-29/`
> ผลของ SVM/KNN เดิมใช้เป็น baseline ในรายงานได้ (Val Macro-F1 0.629 / 0.617, Test Accuracy 0.674 / 0.652)

| ส่วน | เนื้อหา | ผลลัพธ์ |
|---|---|---|
| – | ติดตั้งไลบรารี (ครั้งแรกครั้งเดียว) | แพ็กเกจตาม `requirements.txt` |
| 0 | Config | ตั้งค่าจุดเดียว |
| 1 | Setup + โมดูลที่ใช้ร่วมกับแอป | `fabric_lib.py` |
| 2 | Load + EDA | `results/manifest.csv` + กราฟ |
| 3 | Clean + Group-stratified split | `results/split.csv` |
| 4 | ตั้งค่าโมเดลและฟังก์ชันเทรนร่วม | – |
| 5 | โมเดล 1: ResNet50 (โมเดลเดิม) | `models/candidates/resnet50_s<seed>.pth` |
| 6 | โมเดล 2: ConvNeXt V2-Tiny | `models/candidates/convnextv2_tiny_s<seed>.pth` |
| 7 | โมเดล 3: DINOv3 ViT-S/16 | `models/candidates/dinov3_vits16_s<seed>.pth` |
| 8 | เปรียบเทียบ + เลือกโมเดล + ประเมินบน Test | `results/results.csv` |
| 9 | Error analysis | กราฟรูปที่ทายผิด |
| 10 | Export | `models/best_model.pth` + `models/model_meta.json` |
| 11 | เขียนแอป | `app.py` |
| 12 | เปิดแอป | http://localhost:8501 |

**งานวิจัยอ้างอิง**
- **[R1]** Zhu, Y., Zhao, L., Chen, X., Li, Y., & Wang, J. (2023). *Identification of cashmere and wool based on LBP and GLCM texture feature selection.* Journal of Engineered Fibers and Fabrics, 18. → ที่มาของ baseline SVM (เวอร์ชันก่อนหน้า)
- **[R2]** Mudaffarsyah, M., et al. (2024). *Klasifikasi Jenis Lamun Menggunakan Ekstraksi Fitur GLCM dan Algoritma KNN.* Jurnal Sustainable, 13(2), 17–22. → ที่มาของ baseline KNN (เวอร์ชันก่อนหน้า)
- **[R3]** Enow Gnoupa, M. G., Augousti, A. T., Duran, O., Lanets, O., & Liaskovska, S. (2025). *Multimodal Feature Inputs Enable Improved Automated Textile Identification.* Textiles, 5(3), 31. → ResNet50 (โมเดล 1) ซึ่งงานนี้พบว่าดีที่สุดในกลุ่ม InceptionV3 / ResNet50_V2 / DenseNet / VGG-19
- **[R4]** Woo, S., et al. (2023). *ConvNeXt V2: Co-designing and Scaling ConvNets with Masked Autoencoders.* CVPR 2023. → โมเดล 2
- **[R5]** Siméoni, O., et al. (2025). *DINOv3.* Meta AI. → โมเดล 3

## ติดตั้งไลบรารี

Cell นี้ติดตั้งแพ็กเกจที่ notebook และแอปต้องใช้ลงใน Python ของ kernel ที่เปิดอยู่ ตรวจก่อนว่าขาดอะไร ถ้าครบแล้วจะข้ามทันที จึงรันกับ Run All ได้ทุกครั้ง
- ติดตั้งตาม `requirements.txt` (เวอร์ชันเดียวกับที่ทดสอบไว้)
- เครื่องที่มี GPU NVIDIA จะได้ PyTorch แบบ CUDA (ดาวน์โหลดประมาณ 2.6 GB ครั้งแรกใช้เวลาหลายนาที) เครื่องที่ไม่มีจะได้แบบ CPU
- ถ้าเครื่องมี GPU แต่ PyTorch ที่ติดตั้งอยู่เป็นแบบ CPU จะติดตั้ง PyTorch แบบ CUDA ทับให้ (ตั้ง `FIX_CPU_TORCH = False` เพื่อปิด)
- แนะนำให้ใช้ virtual environment แยกของโปรเจค (ดู README.md) เพื่อไม่ให้ไปแก้ Python หลักของเครื่อง

ถ้าเปิด notebook ด้วย kernel ใหม่ (เพิ่งเปิดหรือเพิ่ง restart) หลังติดตั้งเสร็จจะทำงานต่อได้เลย แต่ถ้า kernel เคยรัน cell อื่นไปแล้วและมีการติดตั้งทับ cell จะหยุดและบอกให้ **Restart kernel แล้ว Run All อีกครั้ง**

In [ ]:
# ===== ติดตั้งไลบรารี: ครั้งแรกจะติดตั้ง ครั้งต่อไปจะข้ามเองถ้าครบแล้ว =====
import importlib, importlib.util, re, shutil, subprocess, sys
from pathlib import Path

FIX_CPU_TORCH = True   # มี GPU NVIDIA แต่ torch เป็นแบบ CPU -> ติดตั้งแบบ CUDA ทับ

assert sys.version_info >= (3, 11), f"ต้องใช้ Python 3.11 ขึ้นไป (ตอนนี้ {sys.version.split()[0]})"

# ชื่อที่ import -> ชื่อแพ็กเกจใน pip
REQUIRED = {"torch": "torch", "torchvision": "torchvision", "numpy": "numpy", "pandas": "pandas",
            "sklearn": "scikit-learn", "scipy": "scipy", "timm": "timm", "transformers": "transformers",
            "PIL": "pillow", "joblib": "joblib", "tqdm": "tqdm", "matplotlib": "matplotlib",
            "seaborn": "seaborn", "streamlit": "streamlit", "ipywidgets": "ipywidgets"}
HAS_NVIDIA = shutil.which("nvidia-smi") is not None


def pip_install(args):
    """รัน pip ด้วย Python ของ kernel นี้ และแสดงผลใน notebook ทีละบรรทัด"""
    cmd = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--progress-bar", "off", *args]
    print("$", " ".join(cmd[2:]))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if not line.startswith(("  Downloading", "  Using cached", "Requirement already satisfied")):
            print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError("pip install ไม่สำเร็จ ดูข้อความด้านบน")


def requirements_file():
    """requirements.txt ของโปรเจค (ถ้าไม่มี GPU NVIDIA ใช้ torch แบบ CPU จาก PyPI แทน)"""
    req = Path("requirements.txt")
    if not req.exists():
        raise FileNotFoundError("ไม่พบ requirements.txt ให้เปิด notebook จากโฟลเดอร์โปรเจค")
    if HAS_NVIDIA:
        return req
    text = req.read_text(encoding="utf-8")
    text = re.sub(r"\+cu\d+", "", text)
    text = "\n".join(l for l in text.splitlines() if "download.pytorch.org" not in l)
    cpu_req = Path("cache") / "requirements_cpu.txt"
    cpu_req.parent.mkdir(exist_ok=True)
    cpu_req.write_text(text, encoding="utf-8")
    return cpu_req


reinstalled = []   # ชื่อ module ที่ติดตั้งหรือติดตั้งทับใน cell นี้
missing_mods = [mod for mod in REQUIRED if importlib.util.find_spec(mod) is None]
if missing_mods:
    print("ยังไม่มีแพ็กเกจ:", ", ".join(REQUIRED[m] for m in missing_mods))
    print("กำลังติดตั้งตาม requirements.txt" + (" (PyTorch แบบ CUDA)" if HAS_NVIDIA else " (PyTorch แบบ CPU)"))
    pip_install(["-r", str(requirements_file())])
    reinstalled = missing_mods
elif HAS_NVIDIA and FIX_CPU_TORCH:
    probe = subprocess.run([sys.executable, "-c", "import torch; print(torch.version.cuda)"],
                           capture_output=True, text=True)
    # ถือว่าเป็นแบบ CPU เฉพาะเมื่อ import สำเร็จและ torch.version.cuda เป็น None
    # (ถ้า import ไม่สำเร็จ ให้การตรวจด้านล่างรายงานสาเหตุ ไม่ติดตั้งทับโดยไม่จำเป็น)
    if probe.returncode == 0 and probe.stdout.strip().splitlines()[-1:] == ["None"]:
        if "torch" in sys.modules:
            raise SystemExit("ต้องติดตั้ง PyTorch แบบ CUDA ทับ แต่ kernel นี้ import torch ไว้แล้ว "
                             "(Windows ลบไฟล์ที่กำลังใช้อยู่ไม่ได้): ให้ Restart kernel แล้วรัน cell นี้เป็น cell แรก")
        print("มี GPU NVIDIA แต่ PyTorch ที่ติดตั้งอยู่เป็นแบบ CPU: ติดตั้งแบบ CUDA ทับ")
        pins = [l.split("#")[0].strip() for l in Path("requirements.txt").read_text(encoding="utf-8").splitlines()]
        pins = [p for p in pins if p.startswith(("torch==", "torchvision==", "--extra-index-url"))]
        extra = [p.split()[1] for p in pins if p.startswith("--extra-index-url")]
        pip_install(["--force-reinstall", "--no-deps", *[a for u in extra for a in ("--index-url", u)],
                     *[p for p in pins if not p.startswith("--")]])
        reinstalled = ["torch", "torchvision"]

importlib.invalidate_caches()

# ทดลอง import โมดูลที่มีไฟล์ DLL ใน process แยก: Windows (Smart App Control / Application Control)
# อาจบล็อก DLL ของ venv ที่อยู่ในโฟลเดอร์ OneDrive ทั้งที่ไฟล์เดียวกันนอก OneDrive ใช้ได้ปกติ
check = subprocess.run([sys.executable, "-c", "import numpy, sklearn.linear_model, scipy.stats, torch, timm, transformers"],
                       capture_output=True, text=True)
if check.returncode != 0:
    err = check.stderr.strip().splitlines()[-1] if check.stderr.strip() else "unknown error"
    hint = ""
    if "Application Control" in err or "WinError 4551" in err or "scipy" in err and "broken" in err:
        hint = ("\nWindows Smart App Control (หรือ Application Control policy) บล็อกไฟล์ DLL ของไลบรารี Python "
                "ซึ่งไม่ใช่ปัญหาของโค้ด ดูวิธีแก้ในหัวข้อ 'ปัญหา Smart App Control' ของ README.md")
    raise ImportError(f"import ไลบรารีไม่สำเร็จด้วย {sys.executable}:\n{err}{hint}")

stale = [m for m in reinstalled if m in sys.modules]
if stale:
    # kernel นี้โหลดเวอร์ชันเก่าไว้ในหน่วยความจำแล้ว ต้องเริ่ม kernel ใหม่จึงจะใช้เวอร์ชันที่เพิ่งติดตั้ง
    raise SystemExit(f"ติดตั้งเสร็จแล้ว แต่ kernel นี้ import {', '.join(stale)} ไว้ก่อน: "
                     "ให้ Restart kernel แล้ว Run All อีกครั้ง")
if reinstalled:
    print("\nติดตั้งเสร็จแล้ว ทำงานต่อได้เลย")
print(f"ไลบรารีครบแล้ว (Python {sys.version.split()[0]}, GPU NVIDIA: {'มี' if HAS_NVIDIA else 'ไม่มี'})")

## 0. Config

ค่าที่ต้องปรับตามเครื่องอยู่ใน cell นี้ที่เดียว ส่วนอื่นของ notebook ไม่ต้องแก้

**โครงสร้างโฟลเดอร์ข้อมูลที่ต้องการ** (ชื่อโฟลเดอร์คลาสไม่สนตัวพิมพ์เล็ก/ใหญ่ และรูปจะอยู่ในโฟลเดอร์ย่อยก็ได้)
```
dataset/
  iBUG/{Cotton,Polyester,Silk,Viscose}/...
  TextileClass7/{Cotton,Polyester,Silk,Viscose}/...
```

**ID ชิ้นผ้า (สำหรับ group split):** ถ้ารูปอยู่ในโฟลเดอร์ย่อยใต้โฟลเดอร์คลาส จะถือว่าโฟลเดอร์ย่อยนั้นคือ 1 ชิ้นผ้า ถ้ารูปวางรวมกันในโฟลเดอร์คลาส ให้ตั้ง `GROUP_REGEX` เพื่อตัดส่วนท้ายชื่อไฟล์ที่ต่างกันระหว่างรูปของชิ้นเดียวกันออก (เช่นเลขมุมแสง)

In [ ]:
# ===== Config: แก้เฉพาะ cell นี้ =====
from pathlib import Path

PROJECT_DIR = Path.cwd()                 # โฟลเดอร์ที่มี notebook นี้ (ให้เปิด notebook จากโฟลเดอร์โปรเจค)
DATA_DIR    = PROJECT_DIR / "dataset"    # dataset/<source>/<Class>/...
OUTPUT_DIR  = PROJECT_DIR                # ที่เก็บผลลัพธ์ทั้งหมด

SOURCES = ["iBUG", "TextileClass7"]      # ชื่อโฟลเดอร์แหล่งข้อมูลใน DATA_DIR
# regex ที่ตัดออกจากท้ายชื่อไฟล์เพื่อได้ ID ชิ้นผ้า (None = 1 ไฟล์ต่อ 1 ชิ้น)
# เช่น r"_\d+$" ทำให้ "sample12_1.png" ... "sample12_4.png" เป็นชิ้นเดียวกัน ("sample12")
# TextileClass7: Roboflow ทำ augmentation ไว้ 3 เวอร์ชันต่อรูปต้นฉบับ ชื่อไฟล์เป็น "<ต้นฉบับ>_jpg.rf.<hash>.jpg"
# จึงตัด "_jpg.rf.<hash>" ออกเพื่อให้ทุกเวอร์ชันของรูปเดียวกันอยู่ชุดเดียวกัน
GROUP_REGEX = {"iBUG": None, "TextileClass7": r"_(jpe?g|png)\.rf\.[0-9a-f]+$"}

# ผลการตรวจคุณภาพข้อมูล (สร้างด้วย data_audit/audit_textileclass7.py) ถ้าไม่มีไฟล์จะข้ามขั้นนั้น
FABRIC_GROUPS_CSV = PROJECT_DIR / "data_audit" / "fabric_groups.csv"  # รูปหลายรูปของผ้าชิ้นเดียวกัน -> ID ชิ้นผ้าเดียวกัน
EXCLUDE_CSV       = PROJECT_DIR / "data_audit" / "exclude.csv"        # รูปที่ label ขัดแย้ง (ผ้าชิ้นเดียวกันแต่ติดคนละคลาส)

# Train ใช้ทุกรูปของแต่ละชิ้น (iBUG 4 แสง, TextileClass7 3 เวอร์ชัน) แต่ Val/Test ใช้ 1 รูปต่อชิ้น
# เพื่อให้ทุกตัวอย่างที่ใช้วัดผลเป็นอิสระต่อกัน (ไม่นับชิ้นเดียวกันซ้ำ 3-4 ครั้ง)
EVAL_ONE_PER_PIECE = True

SEED     = 42
IMG_SIZE = 224
MIN_SIDE = 64                            # ลบรูปที่ด้านใดด้านหนึ่งเล็กกว่านี้ (พิกเซล)
SPLIT    = (0.70, 0.15, 0.15)            # Train / Val / Test

# ===== โมเดลที่เทียบกัน (3 ตัว) =====
# ResNet50 = โมเดลเดิมของงานนี้ (ใช้สูตรเดิมทุกอย่าง)  ConvNeXt V2-Tiny และ DINOv3 ViT-S/16 = โมเดลที่ใหม่กว่า
MODEL_KEYS = ["resnet50", "convnextv2_tiny", "dinov3_vits16"]
SEEDS = (42,)                            # ใส่หลาย seed เช่น (42, 7, 2024) เพื่อรายงาน mean ± std (ใช้เวลาเพิ่มเป็นเท่าตัว)
BATCH_SIZE      = 32                     # ถ้า GPU out-of-memory จะลดเป็น 16 ให้อัตโนมัติ
MAX_EPOCHS      = 30                     # learning rate ลดแบบ cosine จนเป็น 0 ที่ epoch สุดท้าย
PATIENCE        = 7                      # early stopping
LABEL_SMOOTHING = 0.0
STRONG_AUG      = False                  # True = crop กว้างขึ้น + vertical flip + grayscale + random erasing
MIXUP_CUTMIX    = True                   # ผสมภาพ 2 รูปใน batch (MixUp หรือ CutMix) ใช้ label แบบ soft ทนต่อ label ที่ผิดได้ดีกว่า
NUM_WORKERS     = 4

# hyperparameter รายโมเดล: ResNet50 = สูตรเดิมที่ได้ Val Macro-F1 สูงสุดในการทดลองก่อนหน้า
# ส่วน ConvNeXt V2 และ DINOv3 เป็นค่าตั้งต้นที่นิยมสำหรับ fine-tune (ยังไม่ได้จูนบนข้อมูลชุดนี้)
MODEL_CFGS = {
    "resnet50":        dict(lr_backbone=1e-4, lr_head=1e-4, freeze_epochs=0, weight_decay=1e-4, dropout=0.3),
    "convnextv2_tiny": dict(lr_backbone=5e-5, lr_head=5e-4, freeze_epochs=1, weight_decay=0.05, dropout=0.3),
    "dinov3_vits16":   dict(lr_backbone=2e-5, lr_head=5e-4, freeze_epochs=1, weight_decay=0.05, dropout=0.1),
}

N_JOBS = -1                              # จำนวน process ของ joblib (-1 = ทุก core)
SKIP_TRAINING_IF_EXISTS = True           # True = ถ้ามีโมเดลที่เทรนจากข้อมูลชุดเดียวกันอยู่แล้ว ให้โหลดมาใช้แทนการเทรนใหม่
CONF_THRESHOLD = 0.60                    # ความมั่นใจต่ำกว่านี้ แอปจะเตือนให้คนตรวจซ้ำ
LAUNCH_APP = True                        # เปิดแอป Streamlit ตอนท้าย notebook
APP_PORT   = 8501

# โฟลเดอร์ผลลัพธ์ (ไม่ต้องแก้)
CACHE_DIR     = OUTPUT_DIR / "cache"             # features.npz
MODELS_DIR    = OUTPUT_DIR / "models"            # โมเดลที่ดีที่สุด + model_meta.json
CANDIDATE_DIR = MODELS_DIR / "candidates"        # <model>_s<seed>.pth
RESULTS_DIR   = OUTPUT_DIR / "results"           # manifest, split, ตารางผล, กราฟ
for d in (CACHE_DIR, MODELS_DIR, CANDIDATE_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 1. Setup

### 1.1 โมดูลที่ใช้ร่วมกับแอป (`fabric_lib.py`)

การแปลงภาพสำหรับโมเดล การสร้างโมเดลทั้ง 3 ตัว และการโหลดโมเดลที่ดีที่สุด ถูกเขียนไว้ในไฟล์เดียว ทั้ง notebook และ `app.py` import จากไฟล์นี้ เพื่อให้ preprocessing ตอนเทรนกับในแอปตรงกันเสมอ (ลดความเสี่ยง "ผลในแอปแย่กว่าใน notebook") และยังทำให้ DataLoader ใช้หลาย process บน Windows ได้

- **`MODEL_ZOO`**: รายชื่อโมเดลและวิธีโหลด (ResNet50 จาก torchvision, ConvNeXt V2-Tiny จาก `timm`, DINOv3 ViT-S/16 จาก `transformers`)
- **`build_model`**: โหลดโมเดลที่ pretrain แล้วเปลี่ยนหัวจำแนกเป็น 4 คลาส (ตอน inference ในแอปสร้างโครงเปล่าแล้ว load weights ที่บันทึกไว้ ไม่ต้องใช้อินเทอร์เน็ต)
- **`build_transforms`**: resize → center crop 224 → normalize แบบ ImageNet (ตอนเทรนเพิ่ม random crop, หมุน 90°, flip, color jitter)

In [ ]:
%%writefile fabric_lib.py
"""โค้ดที่ใช้ร่วมกันระหว่าง fabric_classification.ipynb และ app.py

ไฟล์นี้ถูกสร้างจาก notebook (ส่วน 1.1) ให้แก้ใน notebook แล้วรัน cell นั้นใหม่ ไม่ต้องแก้ไฟล์นี้โดยตรง
"""
from __future__ import annotations

import io
import json
import random
from pathlib import Path

import numpy as np
from PIL import Image, ImageOps

CLASSES = ["Cotton", "Polyester", "Silk", "Viscose"]
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


# ---------- โหลดรูป ----------
def load_pil(src) -> Image.Image:
    """path / bytes / PIL.Image -> PIL RGB ที่หมุนตาม EXIF แล้ว"""
    if isinstance(src, Image.Image):
        img = src
    elif isinstance(src, (bytes, bytearray)):
        img = Image.open(io.BytesIO(src))
    else:
        img = Image.open(src)
    img = ImageOps.exif_transpose(img)
    return img.convert("RGB")


# ---------- Augmentation / Dataset ----------
class RandomRot90:
    """หมุน 0/90/180/270 องศาแบบสุ่ม (ลายผ้าไม่มีด้านบนที่แน่นอน และไม่เกิดขอบดำเหมือนการหมุนมุมอื่น)"""

    def __call__(self, img):
        k = random.randint(0, 3)
        return img.rotate(90 * k, expand=True) if k else img


def build_transforms(img_size: int = 224, train: bool = False, strong: bool = False):
    from torchvision import transforms as T

    if train and strong:
        return T.Compose([
            T.RandomResizedCrop(img_size, scale=(0.25, 1.0), ratio=(0.75, 1.33), antialias=True),
            RandomRot90(),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15, hue=0.02),
            T.RandomGrayscale(p=0.05),
            T.ToTensor(),
            T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            T.RandomErasing(p=0.25, scale=(0.02, 0.15)),
        ])
    if train:
        return T.Compose([
            T.RandomResizedCrop(img_size, scale=(0.4, 1.0), ratio=(0.8, 1.25), antialias=True),
            RandomRot90(),
            T.RandomHorizontalFlip(),
            T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
            T.ToTensor(),
            T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return T.Compose([
        T.Resize(int(round(img_size * 256 / 224)), antialias=True),
        T.CenterCrop(img_size),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])


class FabricDataset:
    """Dataset แบบ map-style ที่โหลดรูปจากดิสก์ทีละรูป (ไม่โหลดทั้งหมดเข้า RAM)"""

    def __init__(self, paths, labels, transform):
        self.paths = [str(p) for p in paths]
        self.labels = [int(y) for y in labels]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        return self.transform(load_pil(self.paths[i])), self.labels[i]


# ---------- โมเดล: ResNet50 (เดิม) + ConvNeXt V2-Tiny + DINOv3 ViT-S/16 ----------
# family: torchvision = ใช้ torchvision, timm = ใช้ไลบรารี timm, hf = ใช้ transformers (Hugging Face)
MODEL_ZOO = {
    "resnet50": {"name": "ResNet50", "family": "torchvision"},
    "convnextv2_tiny": {"name": "ConvNeXt V2-Tiny", "family": "timm",
                        # ลองตามลำดับ: weights ที่ pretrain บน ImageNet-22k แล้ว fine-tune บน 1k ก่อน
                        "timm_names": ["convnextv2_tiny.fcmae_ft_in22k_in1k", "convnextv2_tiny.fcmae_ft_in1k"]},
    "dinov3_vits16": {"name": "DINOv3 ViT-S/16", "family": "hf",
                      "repo": "facebook/dinov3-vits16-pretrain-lvd1689m",   # gated: ต้องล็อกอิน Hugging Face
                      "fallback_repo": "facebook/dinov2-small"},            # ไม่ gated ใช้เมื่อเข้า DINOv3 ไม่ได้
}
CNN_ARCHS = {k: v["name"] for k, v in MODEL_ZOO.items()}  # ชื่อเดิม (เข้ากันได้กับโค้ดเก่า)


def _hf_classifier_cls():
    """สร้างคลาสหัวจำแนกของ Vision Transformer จาก transformers (import torch เมื่อใช้งานจริงเท่านั้น)"""
    import torch
    import torch.nn as nn

    class HFClassifier(nn.Module):
        """backbone ของ Hugging Face + หัวจำแนก: ใช้ CLS token ต่อกับค่าเฉลี่ยของ patch tokens"""

        def __init__(self, backbone, num_classes, dropout=0.1):
            super().__init__()
            self.backbone = backbone
            hidden = backbone.config.hidden_size
            self.n_special = 1 + int(getattr(backbone.config, "num_register_tokens", 0) or 0)
            self.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden * 2, num_classes))

        def forward(self, x):
            h = self.backbone(pixel_values=x).last_hidden_state
            feat = torch.cat([h[:, 0], h[:, self.n_special:].mean(dim=1)], dim=1)
            return self.fc(feat)

    return HFClassifier


def build_model(arch: str = "resnet50", num_classes: int = len(CLASSES), pretrained: bool = True,
                dropout: float = 0.3, hf_repo: str | None = None, hf_config: dict | None = None):
    """โมเดลที่เปลี่ยนหัวจำแนกเป็น Dropout + Linear(num_classes)

    pretrained=True  : ดาวน์โหลด weights ที่ pretrain มา (ใช้ตอนเทรน)
    pretrained=False : สร้างโครงเปล่า แล้วค่อย load_state_dict (ใช้ตอน inference ในแอป ไม่ต้องใช้อินเทอร์เน็ต)
                       สำหรับโมเดลตระกูล hf ให้ส่ง hf_config (dict) ที่บันทึกไว้ตอนเทรน
    """
    import torch.nn as nn

    if arch not in MODEL_ZOO:
        raise ValueError(f"ไม่รู้จัก arch {arch!r} (รองรับ {list(MODEL_ZOO)})")
    family = MODEL_ZOO[arch]["family"]

    if family == "torchvision":
        from torchvision import models

        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
        model.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(model.fc.in_features, num_classes))

    elif family == "timm":
        import timm

        names = MODEL_ZOO[arch]["timm_names"]
        last_err = None
        for tm_name in (names if pretrained else names[:1]):
            try:
                model = timm.create_model(tm_name, pretrained=pretrained, num_classes=num_classes,
                                          drop_rate=dropout, drop_path_rate=0.1 if pretrained else 0.0)
                break
            except Exception as e:  # ดาวน์โหลด weights ไม่ได้ -> ลองชื่อถัดไป
                last_err = e
        else:
            raise RuntimeError(f"โหลด {arch} จาก timm ไม่สำเร็จ: {last_err}")

    else:  # hf
        from transformers import AutoModel

        cls = _hf_classifier_cls()
        if pretrained:
            backbone = AutoModel.from_pretrained(hf_repo or MODEL_ZOO[arch]["repo"])
        else:
            if not hf_config:
                raise ValueError("ต้องส่ง hf_config (บันทึกอยู่ใน model_meta.json) เพื่อสร้างโครงโมเดล")
            from transformers.models.auto.configuration_auto import CONFIG_MAPPING

            cfg = CONFIG_MAPPING[hf_config["model_type"]].from_dict(dict(hf_config))
            backbone = AutoModel.from_config(cfg)
        model = cls(backbone, num_classes, dropout)
    return model


build_cnn = build_model  # ชื่อเดิม


def model_head(model):
    """หัวจำแนกที่สร้างใหม่ (ไม่ได้ pretrained) ของโมเดลจาก build_model"""
    if hasattr(model, "get_classifier"):  # timm
        return model.get_classifier()
    return model.fc  # torchvision ResNet และ HFClassifier


cnn_head = model_head  # ชื่อเดิม


def hf_config_of(model) -> dict | None:
    """config ของ backbone (ใช้บันทึกลง checkpoint) หรือ None ถ้าไม่ใช่โมเดลตระกูล hf"""
    return model.backbone.config.to_dict() if hasattr(model, "backbone") else None


# ---------- Predictor: ใช้ทั้งตอนจับเวลาใน notebook และในแอป ----------
class TorchPredictor:
    kind = "torch"

    def __init__(self, model, device, img_size: int = 224, classes=CLASSES, batch_size: int = 32):
        import torch

        self.device = torch.device(device)
        self.model = model.to(self.device).eval()
        self.tf = build_transforms(img_size, train=False)
        self.classes = list(classes)
        self.batch_size = batch_size

    def predict_proba(self, images, tta: bool = False) -> np.ndarray:
        import torch

        images = list(images)
        use_amp = self.device.type == "cuda"
        out = []
        with torch.inference_mode(), torch.autocast(device_type=self.device.type, dtype=torch.float16, enabled=use_amp):
            for i in range(0, len(images), self.batch_size):
                x = torch.stack([self.tf(load_pil(im)) for im in images[i:i + self.batch_size]]).to(self.device)
                p = torch.softmax(self.model(x).float(), dim=1)
                if tta:
                    p = (p + torch.softmax(self.model(torch.flip(x, dims=[3])).float(), dim=1)) / 2
                out.append(p.cpu().numpy())
        return np.concatenate(out)


def load_predictor(model_dir, device=None):
    """โหลดโมเดลที่ดีที่สุดตาม model_meta.json"""
    import torch

    model_dir = Path(model_dir)
    meta = json.loads((model_dir / "model_meta.json").read_text(encoding="utf-8"))
    if meta.get("model_type") != "torch":
        raise ValueError("model_meta.json นี้เป็นโมเดลแบบเก่า (SVM/KNN) ให้รัน notebook ส่วนที่ 10 (Export) ใหม่")
    classes = meta["classes"]
    pre = meta["preprocessing"]
    model = build_model(pre.get("arch", "resnet50"), len(classes), pretrained=False, hf_config=pre.get("hf_config"))
    model.load_state_dict(torch.load(model_dir / meta["model_file"], map_location="cpu", weights_only=True))
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    pred = TorchPredictor(model, device, pre["img_size"], classes)
    pred.meta = meta
    return pred

### 1.2 Import, fix seed และตรวจ GPU

ตั้ง seed = 42 ให้ทุกไลบรารี และแสดงเวอร์ชันไลบรารีกับชื่อ GPU เพื่อยืนยันว่าใช้ CUDA ได้ ฟังก์ชัน `evaluate` ด้านล่างใช้วัดผลทุกโมเดลด้วยเกณฑ์เดียวกัน (Accuracy, Precision/Recall/F1 แบบ macro และรายคลาส)

In [ ]:
import sys, os, json, time, random, re, hashlib, warnings, platform, importlib, importlib.util, io

_missing = [m for m in ("numpy", "pandas", "matplotlib", "seaborn", "sklearn", "scipy", "PIL", "torch",
                        "torchvision", "timm", "transformers", "joblib", "tqdm") if importlib.util.find_spec(m) is None]
if _missing:
    raise ModuleNotFoundError(f"Python ของ kernel นี้ ({sys.executable}) ยังไม่มี {', '.join(_missing)}: "
                              "รัน cell 'ติดตั้งไลบรารี' ที่ด้านบนสุดของ notebook ก่อน แล้วรัน cell นี้อีกครั้ง")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import joblib
from joblib import Parallel, delayed
import sklearn, PIL
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import torch, torchvision, timm, transformers
from IPython.display import display, HTML

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))  # ให้ process ลูกของ joblib/DataLoader import fabric_lib ได้
import fabric_lib as fl
importlib.reload(fl)

CLASSES = fl.CLASSES
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def evaluate(y_true, proba):
    """วัดผลจาก probability (n, 4) -> dict ของตัวชี้วัด

    ค่า macro และ min_recall คิดเฉพาะคลาสที่มีตัวอย่างจริงในชุดนั้น (เช่นเมื่อแยกตามแหล่งข้อมูลแล้วบางคลาสไม่มีรูป)
    คลาสที่ไม่มีตัวอย่างจะได้ค่ารายคลาสเป็น NaN แทน 0 ที่ทำให้ค่าเฉลี่ยต่ำเกินจริง
    """
    y_pred = np.asarray(proba).argmax(1)
    p, r, f, support = precision_recall_fscore_support(y_true, y_pred, labels=range(len(CLASSES)), zero_division=0)
    has = support > 0
    p, r, f = (np.where(has, v, np.nan) for v in (p, r, f))
    m = {"accuracy": accuracy_score(y_true, y_pred), "macro_precision": np.nanmean(p),
         "macro_recall": np.nanmean(r), "macro_f1": np.nanmean(f), "min_recall": np.nanmin(r)}
    for i, c in enumerate(CLASSES):
        m[f"precision_{c}"], m[f"recall_{c}"], m[f"f1_{c}"] = p[i], r[i], f[i]
    return {k: float(v) for k, v in m.items()}


def savefig(fig, name):
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / name, dpi=150, bbox_inches="tight")


seed_everything()
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100
warnings.filterwarnings("ignore", message=".*least populated class.*")

versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "scikit-learn": sklearn.__version__, "pillow": PIL.__version__, "torch": torch.__version__,
            "torchvision": torchvision.__version__, "timm": timm.__version__, "transformers": transformers.__version__}
display(pd.Series(versions, name="version").to_frame())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} ({props.total_memory / 2**30:.1f} GB), CUDA {torch.version.cuda}")
else:
    print("ไม่พบ GPU ที่ใช้ CUDA ได้: จะเทรน CNN บน CPU ซึ่งช้ามาก ตรวจการติดตั้ง PyTorch แบบ CUDA ใน README.md")

## 2. Load + EDA

สร้างตาราง manifest (path, class, source, sample_id) จากโฟลเดอร์ข้อมูล แล้วเปิดอ่านทุกรูปหนึ่งครั้งเพื่อเก็บขนาดภาพ ตรวจไฟล์เสีย และคำนวณ hash (ใช้ต่อในส่วนที่ 3) ผลการสแกนถูกเก็บไว้ใน `manifest.csv` รันซ้ำแล้วไม่ต้องสแกนใหม่

ข้อมูล: **iBUG Fabrics** (Kampouris et al., ECCV 2016; ชุดเดียวกับที่ R3 ใช้) และ **TextileClass-7** (Mendeley Data)

In [ ]:
def piece_id(source, cls, rel):
    """ID ชิ้นผ้าสำหรับ group split: โฟลเดอร์ย่อย หรือชื่อไฟล์ที่ตัดส่วนท้ายด้วย GROUP_REGEX"""
    if len(rel.parts) > 1:
        piece = "/".join(rel.parts[:-1])
    else:
        rx = GROUP_REGEX.get(source)
        piece = re.sub(rx, "", rel.stem) if rx else rel.stem
    return f"{source}/{cls}/{piece}"


rows, missing = [], []
for source in SOURCES:
    src_dir = DATA_DIR / source
    if not src_dir.is_dir():
        missing.append(src_dir)
        continue
    class_dirs = {d.name.lower(): d for d in src_dir.iterdir() if d.is_dir()}
    for cls in CLASSES:
        cdir = class_dirs.get(cls.lower())
        if cdir is None:
            missing.append(src_dir / cls)
            continue
        for p in sorted(cdir.rglob("*")):
            if p.is_file() and p.suffix.lower() in fl.IMG_EXTS:
                rows.append({"path": p.relative_to(DATA_DIR).as_posix(), "class": cls, "source": source,
                             "sample_id": piece_id(source, cls, p.relative_to(cdir))})
if missing:
    print("ไม่พบโฟลเดอร์:", *missing, sep="\n  ")
if not rows:
    raise FileNotFoundError(f"ไม่พบรูปใน {DATA_DIR} ดูโครงสร้างโฟลเดอร์ที่ต้องการใน cell Config")
manifest = pd.DataFrame(rows)
print(f"พบ {len(manifest):,} รูป")

if FABRIC_GROUPS_CSV.exists():
    fabric_of = pd.read_csv(FABRIC_GROUPS_CSV).set_index("photo_group")["fabric_id"]
    before = manifest["sample_id"].nunique()
    manifest["sample_id"] = manifest["sample_id"].map(fabric_of).fillna(manifest["sample_id"])
    print(f"รวมรูปของผ้าชิ้นเดียวกันตาม {FABRIC_GROUPS_CSV.name}: {before:,} -> {manifest['sample_id'].nunique():,} ชิ้น")

In [ ]:
MANIFEST_PATH = RESULTS_DIR / "manifest.csv"
SCAN_COLS = ["md5", "width", "height", "ok", "error"]


def scan_image(path):
    """อ่านไฟล์ทั้งไฟล์: md5 ของไฟล์ + ขนาด + ถอดรหัสภาพได้หรือไม่"""
    data = Path(path).read_bytes()
    md5 = hashlib.md5(data).hexdigest()
    try:
        with Image.open(io.BytesIO(data)) as im:
            im.load()
            w, h = im.size
        return md5, w, h, True, ""
    except Exception as e:
        return md5, 0, 0, False, type(e).__name__


scanned = False
if MANIFEST_PATH.exists():
    old = pd.read_csv(MANIFEST_PATH, keep_default_na=False)
    if set(SCAN_COLS) <= set(old.columns) and set(old.path) == set(manifest.path):
        manifest = manifest.merge(old[["path"] + SCAN_COLS], on="path")
        manifest["ok"] = manifest["ok"].astype(str).eq("True")
        scanned = True
        print("ใช้ผลสแกนเดิมจาก", MANIFEST_PATH.name)
if not scanned:
    res = Parallel(n_jobs=N_JOBS, batch_size=32)(
        delayed(scan_image)(DATA_DIR / p) for p in tqdm(manifest.path, desc="scan images"))
    manifest[SCAN_COLS] = pd.DataFrame(res, columns=SCAN_COLS)
    manifest.to_csv(MANIFEST_PATH, index=False)
    print("บันทึก", MANIFEST_PATH)
display(manifest.head())

In [ ]:
counts = pd.crosstab(manifest["class"], manifest["source"]).reindex(CLASSES).fillna(0).astype(int)
table = counts.copy()
table["Total"] = table.sum(axis=1)
table["Share"] = (table["Total"] / table["Total"].sum()).map("{:.1%}".format)
table.loc["Total"] = list(counts.sum()) + [counts.values.sum(), "100%"]
display(table)

fig, ax = plt.subplots(figsize=(7, 4))
counts.plot.bar(stacked=True, ax=ax, rot=0)
ax.set(title="Images per class and source", xlabel="", ylabel="images")
savefig(fig, "eda_class_counts.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 5, figsize=(12, 2.6 * len(CLASSES)))
for r, cls in enumerate(CLASSES):
    sub = manifest[(manifest["class"] == cls) & manifest["ok"]]
    pick = sub.sample(min(5, len(sub)), random_state=SEED)
    for c in range(5):
        ax = axes[r, c]
        ax.axis("off")
        if c < len(pick):
            row = pick.iloc[c]
            img = fl.load_pil(DATA_DIR / row.path)
            img.thumbnail((256, 256))
            ax.imshow(img)
            ax.set_title(f"{cls} | {row.source}", fontsize=9)
fig.suptitle("Sample images (5 per class)")
savefig(fig, "eda_samples.png")
plt.show()

In [ ]:
ok = manifest[manifest["ok"]].assign(min_side=lambda d: np.minimum(d.width, d.height))
display(ok.groupby("source")[["width", "height"]].describe().T.round(0))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=ok, x="width", y="height", hue="source", s=12, alpha=0.5, ax=axes[0])
axes[0].set_title("Image size (px)")
sns.histplot(data=ok, x="min_side", hue="source", bins=40, ax=axes[1])
axes[1].axvline(MIN_SIDE, color="red", ls="--", label=f"MIN_SIDE={MIN_SIDE}")
axes[1].set_title("Shorter side (px)")
savefig(fig, "eda_image_sizes.png")
plt.show()

## 3. Clean + Split

**ทำความสะอาด:** ลบภาพที่เปิดไม่ได้, ภาพที่ด้านใดด้านหนึ่งเล็กกว่า `MIN_SIDE` และภาพซ้ำ (md5 ของไฟล์เท่ากัน) ถ้าภาพซ้ำอยู่คนละคลาสจะลบทิ้งทั้งหมดเพราะไม่รู้ว่า label ไหนถูก

**ผลจากการตรวจคุณภาพ TextileClass-7** (`data_audit/`, ดูวิธีใน `audit_textileclass7.py`): ชื่อไฟล์บอกเครื่องและเวลาที่ถ่าย จึงหารูปของผ้าชิ้นเดียวกันได้จาก "เครื่องเดียวกัน + ถ่ายห่างไม่เกิน 10 วินาที + ภาพคล้ายกันมาก (cosine similarity ของ ResNet50 ≥ 0.90)"
- 136 คู่อยู่คลาสเดียวกันแต่ถูกนับเป็นคนละชิ้น → รวมเป็นชิ้นเดียว (1,433 → 1,308 ชิ้น) ตาม `fabric_groups.csv` เพื่อไม่ให้รั่วข้าม Train/Test
- 5 คู่เป็นผ้าชิ้นเดียวกันแต่ติด label คนละคลาส → ตัดออกทั้งสองฝั่งตาม `exclude.csv` (ตรวจด้วยตาแล้ว ดู `review_pairs.png`)
- ผ้าลายพิมพ์เดียวกันที่ถ่ายห่างกันเป็นชั่วโมงและสีต่างกัน (B3–B5) เก็บไว้ เพราะน่าจะเป็นลายเดียวกันที่พิมพ์บนผ้าคนละชนิด

**แบ่งข้อมูล 70/15/15 แบบ group-stratified:** ภาพจากชิ้นผ้าเดียวกัน (เช่น iBUG ที่ถ่ายชิ้นเดียวกันภายใต้แสงหลายทิศ) ต้องอยู่ชุดเดียวกันเสมอ และแต่ละชุดมีสัดส่วน class × source ใกล้เคียงกัน ใช้ `StratifiedGroupKFold` 20 fold แล้วรวม 14 fold เป็น Train, 3 เป็น Val, 3 เป็น Test

Augmentation ทำเฉพาะชุด Train ตอนเทรน (ส่วนที่ 4–7) หลังแบ่งแล้วเท่านั้น ต่างจาก R2 ที่ augment ก่อนแบ่งซึ่งทำให้ภาพเกือบซ้ำรั่วเข้าชุด Test

**จำนวนชิ้นจริงน้อยกว่าจำนวนไฟล์:** iBUG ถ่าย 1 ชิ้นภายใต้แสง 4 ทิศ (901 ชิ้น = 3,604 รูป) และ TextileClass-7 ถูก Roboflow augment ไว้แล้ว 3 เวอร์ชันต่อรูปในชุด train เดิม (3,439 ไฟล์ = รูปจริง 1,433 รูป) ถ้า `EVAL_ONE_PER_PIECE = True` Train จะใช้ทุกไฟล์ แต่ Val/Test เก็บไว้ 1 รูปต่อชิ้น (สุ่มด้วย seed) เพื่อไม่ให้ชิ้นเดียวกันถูกนับหลายครั้งตอนวัดผล

In [ ]:
df = manifest.copy()
n0 = len(df)
bad = ~df["ok"]
small = df["ok"] & (np.minimum(df.width, df.height) < MIN_SIDE)
df = df[~bad & ~small]
n_cls_per_md5 = df.groupby("md5")["class"].nunique()
conflict = df["md5"].isin(n_cls_per_md5[n_cls_per_md5 > 1].index)
df = df[~conflict]
dup = df.duplicated("md5", keep="first")
df = df[~dup]
excluded = pd.Series(False, index=df.index)
if EXCLUDE_CSV.exists():
    exclude_groups = set(pd.read_csv(EXCLUDE_CSV)["photo_group"])
    photo_group = df["source"] + "/" + df["class"] + "/" + df["path"].map(
        lambda p: re.sub(GROUP_REGEX.get(p.split("/")[0]) or r"$^", "", Path(p).stem))
    excluded = photo_group.isin(exclude_groups)
    df = df[~excluded]
df = df.reset_index(drop=True)

display(pd.DataFrame({"count": {
    "ภาพทั้งหมด": n0, "เปิดไม่ได้ (เสีย)": int(bad.sum()), f"เล็กกว่า {MIN_SIDE}px": int(small.sum()),
    "ซ้ำแต่คนละคลาส (ลบทั้งหมด)": int(conflict.sum()), "ซ้ำในคลาสเดียวกัน (เก็บไว้ 1)": int(dup.sum()),
    f"label ขัดแย้งตาม {EXCLUDE_CSV.name}": int(excluded.sum()), "คงเหลือ": len(df)}}))

pieces = df.groupby("source").agg(images=("path", "size"), pieces=("sample_id", "nunique"))
pieces["images_per_piece"] = (pieces.images / pieces.pieces).round(2)
display(pieces)
if "iBUG" in pieces.index and pieces.loc["iBUG", "images_per_piece"] <= 1.0:
    warnings.warn("iBUG มี 1 รูปต่อ 1 ชิ้นผ้า ถ้าชิ้นเดียวกันถูกถ่ายหลายรูป (หลายทิศแสง) ให้ตั้ง "
                  "GROUP_REGEX['iBUG'] ใน Config มิฉะนั้นภาพของชิ้นเดียวกันอาจรั่วข้ามชุด Train/Test")

In [ ]:
SPLIT_PATH = RESULTS_DIR / "split.csv"
N_FOLDS = 20

split_df = None
if SPLIT_PATH.exists():
    old = pd.read_csv(SPLIT_PATH)
    # ใช้ของเดิมได้เฉพาะเมื่อรูปและการจัดกลุ่มชิ้นผ้าเหมือนเดิมทุกประการ
    if set(zip(old.path, old.sample_id)) == set(zip(df.path, df.sample_id)):
        split_df = df.merge(old[["path", "split"]], on="path")
        print("ใช้การแบ่งข้อมูลเดิมจาก", SPLIT_PATH.name)
    else:
        print("ข้อมูลเปลี่ยนจาก split.csv เดิม จะแบ่งใหม่")
if split_df is None:
    strata = df["class"] + "|" + df["source"]
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold = np.empty(len(df), dtype=int)
    for k, (_, idx) in enumerate(sgkf.split(df, strata, groups=df["sample_id"])):
        fold[idx] = k
    n_tr, n_va = round(N_FOLDS * SPLIT[0]), round(N_FOLDS * SPLIT[1])
    split_df = df.assign(split=np.select([fold < n_tr, fold < n_tr + n_va], ["train", "val"], "test"))
    split_df[["path", "class", "source", "sample_id", "split"]].to_csv(SPLIT_PATH, index=False)
    print("บันทึก", SPLIT_PATH)

leak = split_df.groupby("sample_id")["split"].nunique()
assert (leak == 1).all(), f"มีชิ้นผ้า {int((leak > 1).sum())} ชิ้นอยู่มากกว่าหนึ่งชุด"
print("ตรวจแล้ว: ไม่มีชิ้นผ้าซ้ำข้ามชุด Train/Val/Test")

if EVAL_ONE_PER_PIECE:
    is_eval = split_df["split"] != "train"
    one = split_df[is_eval].sort_values("path").groupby("sample_id").sample(1, random_state=SEED)
    print(f"Val/Test: เก็บ 1 รูปต่อชิ้น {len(one):,} จาก {int(is_eval.sum()):,} รูป")
    split_df = pd.concat([split_df[~is_eval], one])
split_df = split_df.sort_values("path").reset_index(drop=True)

y_all = split_df["class"].map(CLASS_TO_IDX).to_numpy()
g_all = split_df["sample_id"].to_numpy()
masks = {s: (split_df["split"] == s).to_numpy() for s in ("train", "val", "test")}

summary = pd.crosstab([split_df["source"], split_df["class"]], split_df["split"])[["train", "val", "test"]]
summary.loc[("All", "All"), :] = summary.sum()
display(summary.assign(**{"train %": lambda d: (d.train / d.sum(axis=1)).map("{:.0%}".format)}))

## 4. ตั้งค่าโมเดลและฟังก์ชันเทรนร่วม

เทียบ 3 โมเดลที่ pretrain บน ImageNet (หรือชุดภาพขนาดใหญ่ของตัวเอง) แล้ว fine-tune ทั้งโครงข่ายกับรูปผ้า 224×224 ด้วยชุดข้อมูล การแบ่ง Train/Val/Test และ augmentation เดียวกัน จึงเทียบกันได้อย่างเป็นธรรม

| # | โมเดล | ประเภท | ขนาด | ปี | Pretrain |
|---|---|---|---|---|---|
| 1 | **ResNet50** (เดิม) | CNN แบบ residual | ~25M | 2015 | ImageNet-1k (torchvision V2) |
| 2 | **ConvNeXt V2-Tiny** | CNN ยุคใหม่ (ปรับให้ใกล้ transformer) | ~28M | 2023 | ImageNet-22k → 1k (timm) |
| 3 | **DINOv3 ViT-S/16** | Vision Transformer | ~21M | 2025 | self-supervised บนภาพ 1.7 พันล้านรูป (Hugging Face) |

**สูตรเทรน (ใช้ร่วมกันทั้ง 3 โมเดล)**
- **Augmentation (Train เท่านั้น):** random resized crop, หมุน 90°, flip, color jitter และ MixUp/CutMix (ตั้งใน Config)
- **Loss:** CrossEntropy ถ่วงน้ำหนักตามสัดส่วนคลาส
- **Optimizer:** AdamW, lr ของ backbone กับหัวจำแนกแยกกัน, เทรนเฉพาะหัวจำแนกก่อน `freeze_epochs` epoch, lr ลดแบบ cosine
- **Early stopping:** หยุดเมื่อ Val Macro-F1 ไม่ดีขึ้น 7 epoch แล้วใช้ weights ของ epoch ที่ดีที่สุด (เลือกจาก Val เท่านั้น ไม่ใช้ Test)
- **Mixed precision (AMP)** เพื่อให้พอกับ GPU 6 GB; ถ้า out-of-memory จะลด batch เป็น 16 อัตโนมัติ
- ถ้ามี checkpoint ที่เทรนจากข้อมูลและ hyperparameter ชุดเดียวกันอยู่แล้ว จะโหลดมาใช้แทนการเทรนใหม่ (`SKIP_TRAINING_IF_EXISTS`)

**หมายเหตุ**
- ResNet50 ใช้สูตรเดิมที่คัดเลือกไว้แล้วจากการทดลองลด overfit ในเวอร์ชันก่อน (`results/cnn_experiments.csv`) ส่วนอีก 2 โมเดลใช้ค่าตั้งต้นที่ยังไม่ได้จูน จึงอาจไม่ใช่สูตรที่ดีที่สุดของโมเดลนั้น
- ชุด Val/Test มีราว 330 รูป ความคลาดเคลื่อนทางสถิติราว ±2 จุด ผลที่ต่างกันน้อยกว่านี้บอกไม่ได้ว่าโมเดลไหนดีกว่าจริง จึงมีช่วงความเชื่อมั่นและ McNemar test ในส่วน 8.2
- **DINOv3 เป็นโมเดลแบบ gated:** ต้องล็อกอิน Hugging Face (`hf auth login`) และกดยอมรับเงื่อนไขที่หน้า https://huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m ก่อน ถ้าเข้าไม่ได้ notebook จะใช้ DINOv2 ViT-S/14 (ไม่ gated) แทนและระบุชื่อโมเดลให้ตรงตามจริงในทุกตาราง

In [ ]:
y_tr, y_va, y_te = (y_all[masks[s]] for s in ("train", "val", "test"))

# fingerprint ของข้อมูล: ถ้า split เปลี่ยน โมเดลที่ cache ไว้จะถูกเทรนใหม่
FINGERPRINT = hashlib.md5(json.dumps({"split": split_df[["path", "split"]].values.tolist(),
                                      "seed": SEED}).encode()).hexdigest()[:12]
print(f"train {len(y_tr):,} | val {len(y_va):,} | test {len(y_te):,} | data fingerprint {FINGERPRINT}")

assert set(MODEL_KEYS) == set(fl.MODEL_ZOO), "MODEL_KEYS ใน Config ต้องตรงกับ fl.MODEL_ZOO"
assert set(MODEL_CFGS) == set(MODEL_KEYS), "MODEL_CFGS ต้องมีค่าของทุกโมเดล"

_HF_REPO = {}


def resolve_hf_repo(key):
    """repo ของโมเดลตระกูล Hugging Face: ลอง DINOv3 ก่อน ถ้าเข้าไม่ได้ (gated) ใช้ตัวสำรอง"""
    if fl.MODEL_ZOO[key]["family"] != "hf":
        return None
    if key not in _HF_REPO:
        from transformers import AutoConfig
        spec = fl.MODEL_ZOO[key]
        try:
            AutoConfig.from_pretrained(spec["repo"])
            _HF_REPO[key] = spec["repo"]
        except Exception as e:
            first = str(e).strip().splitlines()[0][:160] if str(e).strip() else type(e).__name__
            print(f"เข้า {spec['repo']} ไม่ได้ ({first})\n"
                  f"-> ล็อกอินด้วย `hf auth login` และกดยอมรับเงื่อนไขที่ https://huggingface.co/{spec['repo']}\n"
                  f"-> ตอนนี้ใช้ {spec['fallback_repo']} แทนชั่วคราว")
            _HF_REPO[key] = spec["fallback_repo"]
    return _HF_REPO[key]


def model_label(key, hf_repo=None):
    if key == "dinov3_vits16" and hf_repo and "dinov3" not in hf_repo:
        return "DINOv2 ViT-S/14 (สำรองแทน DINOv3)"
    return fl.MODEL_ZOO[key]["name"]

In [ ]:
import math
import torch.nn as nn
from torch.utils.data import DataLoader

USE_AMP = DEVICE.type == "cuda"
RUNS, VAL_PROBA, TRAIN_TIME, NAMES = {}, {}, {}, []   # RUNS[(key, seed)] = {model, ckpt, val_proba, label}
KEY_OF, BEST_SEED, SEED_F1 = {}, {}, {}               # ชื่อโมเดล -> key / seed ที่ดีที่สุด / Val Macro-F1 ทุก seed


def hparams_of(key, seed):
    c = MODEL_CFGS[key]
    return {"arch": key, "img_size": IMG_SIZE, "max_epochs": MAX_EPOCHS, "patience": PATIENCE,
            "lr_backbone": c["lr_backbone"], "lr_head": c["lr_head"], "freeze_epochs": c["freeze_epochs"],
            "weight_decay": c["weight_decay"], "dropout": c["dropout"], "label_smoothing": LABEL_SMOOTHING,
            "strong_aug": STRONG_AUG, "mixup_cutmix": MIXUP_CUTMIX, "seed": seed}


def make_loader(split, train, batch_size, seed=SEED):
    m = masks[split]
    ds = fl.FabricDataset([DATA_DIR / p for p in split_df["path"][m]], y_all[m],
                          fl.build_transforms(IMG_SIZE, train=train, strong=STRONG_AUG))
    return DataLoader(ds, batch_size=batch_size, shuffle=train, num_workers=NUM_WORKERS,
                      pin_memory=USE_AMP, persistent_workers=NUM_WORKERS > 0,
                      generator=torch.Generator().manual_seed(seed))


@torch.inference_mode()
def predict_loader(model, loader, criterion=None):
    model.eval()
    probs, total, n = [], 0.0, 0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x)
        logits = logits.float()
        if criterion is not None:
            total += criterion(logits, y.to(DEVICE)).item() * len(y)
            n += len(y)
        probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(probs), (total / n if n else None)


def train_model(key, seed, batch_size, hf_repo):
    """fine-tune โมเดล 1 ตัวด้วย seed ที่กำหนด -> (weights ที่ดีที่สุด, history, วินาทีที่ใช้, hf_config)"""
    hp = hparams_of(key, seed)
    seed_everything(seed)
    train_loader = make_loader("train", True, batch_size, seed)
    val_loader = make_loader("val", False, batch_size, seed)
    counts = np.bincount(y_tr, minlength=len(CLASSES))
    class_w = torch.tensor(counts.sum() / (len(CLASSES) * counts), dtype=torch.float32, device=DEVICE)
    print("class weights:", dict(zip(CLASSES, class_w.cpu().numpy().round(2))))

    model = fl.build_model(key, len(CLASSES), pretrained=True, dropout=hp["dropout"], hf_repo=hf_repo).to(DEVICE)
    head_ids = {id(p) for p in fl.model_head(model).parameters()}
    backbone = [p for p in model.parameters() if id(p) not in head_ids]
    head = [p for p in model.parameters() if id(p) in head_ids]
    optimizer = torch.optim.AdamW([{"params": backbone, "lr": hp["lr_backbone"]}, {"params": head, "lr": hp["lr_head"]}],
                                  weight_decay=hp["weight_decay"])
    base_lrs = [hp["lr_backbone"], hp["lr_head"]]
    criterion = nn.CrossEntropyLoss(weight=class_w, label_smoothing=hp["label_smoothing"])
    eval_criterion = nn.CrossEntropyLoss(weight=class_w)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    mixer = None
    if hp["mixup_cutmix"]:
        from torchvision.transforms import v2
        mixer = v2.RandomChoice([v2.MixUp(alpha=0.2, num_classes=len(CLASSES)),
                                 v2.CutMix(alpha=1.0, num_classes=len(CLASSES))])
    total_steps, step = hp["max_epochs"] * len(train_loader), 0

    history, best_f1, best_state, wait = [], -1.0, None, 0
    t0 = time.perf_counter()
    for epoch in range(1, hp["max_epochs"] + 1):
        frozen = epoch <= hp["freeze_epochs"]
        for p in backbone:
            p.requires_grad_(not frozen)
        model.train()
        tot_loss, correct, n = 0.0, 0, 0
        for x, y in tqdm(train_loader, desc=f"{key} epoch {epoch}/{hp['max_epochs']}", leave=False):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            cosine = 0.5 * (1 + math.cos(math.pi * step / total_steps))
            for group, base in zip(optimizer.param_groups, base_lrs):
                group["lr"] = base * cosine
            step += 1
            target = y
            if mixer is not None:
                x, target = mixer(x, y)  # soft labels (B, 4)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, target)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            tot_loss += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            n += len(y)

        val_proba, val_loss = predict_loader(model, val_loader, eval_criterion)
        vm = evaluate(y_va, val_proba)
        history.append({"epoch": epoch, "train_loss": tot_loss / n, "train_acc": correct / n,
                        "val_loss": val_loss, "val_acc": vm["accuracy"], "val_macro_f1": vm["macro_f1"],
                        "lr": optimizer.param_groups[0]["lr"], "frozen": frozen})
        improved = vm["macro_f1"] > best_f1 + 1e-4
        print(f"epoch {epoch:2d}{' (frozen)' if frozen else '         '} | train loss {tot_loss / n:.3f} "
              f"acc {correct / n:.3f} | val loss {val_loss:.3f} acc {vm['accuracy']:.3f} F1 {vm['macro_f1']:.3f}"
              f"{'  *' if improved else ''} | {time.perf_counter() - t0:.0f}s")
        if improved:
            best_f1, wait = vm["macro_f1"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        elif not frozen:
            wait += 1
            if wait >= hp["patience"]:
                print(f"early stopping (Val Macro-F1 ไม่ดีขึ้น {hp['patience']} epoch)")
                break
    elapsed = time.perf_counter() - t0
    hf_config = fl.hf_config_of(model)
    del model, optimizer
    torch.cuda.empty_cache()
    return best_state, history, elapsed, hf_config


def get_run(key, seed):
    """โหลด checkpoint ถ้ามีอยู่แล้ว (ข้อมูลและ hyperparameter เดียวกัน) ไม่เช่นนั้นเทรนใหม่"""
    hp = hparams_of(key, seed)
    fp = hashlib.md5((FINGERPRINT + json.dumps(hp)).encode()).hexdigest()[:12]
    path = CANDIDATE_DIR / f"{key}_s{seed}.pth"
    ckpt = None
    if SKIP_TRAINING_IF_EXISTS and path.exists():
        old = torch.load(path, map_location="cpu", weights_only=True)
        if old.get("fingerprint") == fp:
            ckpt = old
            print(f"โหลด {key} (seed {seed}) จาก {path.name} (ข้อมูลและ hyperparameter ชุดเดียวกัน)")
    if ckpt is None:
        repo = resolve_hf_repo(key)
        for bs in dict.fromkeys((BATCH_SIZE, 16)):
            try:
                state, history, elapsed, hf_config = train_model(key, seed, bs, repo)
                break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                print(f"GPU out-of-memory ที่ batch {bs} ลอง batch ที่เล็กลง")
        else:
            raise RuntimeError("GPU memory ไม่พอแม้ใช้ batch 16")
        ckpt = {"state_dict": state, "history": history, "train_time_s": elapsed, "batch_size": bs,
                "fingerprint": fp, "hparams": hp, "hf_config": hf_config, "hf_repo": repo}
        torch.save(ckpt, path)
        print("บันทึก", path)

    model = fl.build_model(key, len(CLASSES), pretrained=False, dropout=hp["dropout"], hf_config=ckpt.get("hf_config"))
    model.load_state_dict(ckpt["state_dict"])
    model = model.to(DEVICE).eval()
    val_proba, _ = predict_loader(model, make_loader("val", False, ckpt["batch_size"], seed))
    return {"model": model, "ckpt": ckpt, "val_proba": val_proba, "label": model_label(key, ckpt.get("hf_repo"))}


def run_model(key):
    """เทรน (หรือโหลด) โมเดล key ทุก seed ใน SEEDS แล้วเลือก seed ที่ Val Macro-F1 สูงสุดเป็นตัวแทนของโมเดล"""
    per_seed = {}
    for seed in SEEDS:
        run = get_run(key, seed)
        RUNS[(key, seed)] = run
        m = evaluate(y_va, run["val_proba"])
        per_seed[seed] = m["macro_f1"]
        print(f"{run['label']} seed {seed}: Val Macro-F1 {m['macro_f1']:.3f} | Val Acc {m['accuracy']:.3f} | "
              f"train {run['ckpt']['train_time_s'] / 60:.1f} min, batch {run['ckpt']['batch_size']}")
    best_seed = max(per_seed, key=per_seed.get)
    run = RUNS[(key, best_seed)]
    name = run["label"]
    KEY_OF[name], BEST_SEED[name], SEED_F1[name] = key, best_seed, list(per_seed.values())
    VAL_PROBA[name], TRAIN_TIME[name] = run["val_proba"], run["ckpt"]["train_time_s"]
    if name not in NAMES:
        NAMES.append(name)
    return name


def plot_curves(name):
    key = KEY_OF[name]
    hist = pd.DataFrame(RUNS[(key, BEST_SEED[name])]["ckpt"]["history"])
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    hist.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0], marker=".", title=f"{name}: Loss")
    hist.plot(x="epoch", y=["train_acc", "val_acc"], ax=axes[1], marker=".",
              title="Accuracy" + (" (train acc on MixUp/CutMix images)" if MIXUP_CUTMIX else ""))
    hist.plot(x="epoch", y="val_macro_f1", ax=axes[2], marker=".", title="Val Macro-F1", legend=False)
    best_epoch = int(hist.loc[hist.val_macro_f1.idxmax(), "epoch"])
    for ax in axes:
        ax.axvline(best_epoch, color="gray", ls="--")
    savefig(fig, f"training_curves_{key}.png")
    plt.show()

## 5. โมเดล 1: ResNet50 (โมเดลเดิม) [R3]

เก็บ ResNet50 ไว้เป็นโมเดลอ้างอิง Gnoupa et al. (2025) [R3] พบว่า ResNet50_V2 ให้ผลดีที่สุด (F1 0.913) เมื่อเทียบกับ InceptionV3, DenseNet และ VGG-19 ที่นี่ fine-tune ResNet50 (weights ImageNet1K_V2) ด้วยสูตรเดิมทุกอย่าง (lr 1e-4, dropout 0.3, MixUp/CutMix, cosine) รับภาพ RGB 224×224 โดยตรง ไม่ได้ใช้ late fusion ของ LBP + albedo map แบบใน R3

**ผลการทดลองลด overfit ในเวอร์ชันก่อน** (เลือกจาก Val Macro-F1 เท่านั้น) ดู `results/cnn_experiments.csv` สูตรที่ได้ดีที่สุดคือ ResNet50 + MixUp/CutMix (Val Macro-F1 0.849) ซึ่งเป็นสูตรที่ใช้ที่นี่ และ EfficientNet-B0 ให้ผลใกล้เคียงกัน (0.844) แปลว่าการเปลี่ยนสถาปัตยกรรมอย่างเดียวช่วยได้ไม่มาก ซึ่งสอดคล้องกับที่พบว่า TextileClass-7 มี label ที่ไม่สม่ำเสมอ (ผ้าลายเดียวกันถูกติดคนละคลาส) MixUp/CutMix ช่วยเพราะ soft label ลดการ "จำ" ตัวอย่างที่ label ผิด

รันครั้งแรกจะดาวน์โหลด weights ของ ResNet50 (~100 MB) จาก download.pytorch.org

In [ ]:
RESNET_NAME = run_model("resnet50")
plot_curves(RESNET_NAME)

## 6. โมเดล 2: ConvNeXt V2-Tiny [R4]

CNN ยุคใหม่ที่ออกแบบโครงสร้างใหม่ให้ใกล้ transformer (kernel 7×7 แบบ depthwise, LayerNorm, GELU และ GRN layer) และ pretrain ด้วย masked autoencoder (FCMAE) แล้ว fine-tune บน ImageNet-22k → 1k ขนาดและต้นทุนใกล้เคียง ResNet50 จึงใช้ตอบคำถามว่าสถาปัตยกรรม CNN ที่ใหม่กว่าให้ผลดีกว่าบนข้อมูลชุดนี้หรือไม่

ใช้ผ่านไลบรารี `timm` (`convnextv2_tiny.fcmae_ft_in22k_in1k`, สำรอง `convnextv2_tiny.fcmae_ft_in1k`) เทรนด้วย lr backbone 5e-5, lr หัวจำแนก 5e-4, weight decay 0.05, stochastic depth 0.1 และเทรนเฉพาะหัวจำแนก 1 epoch แรก ถ้า GPU 6 GB ไม่พอที่ batch 32 จะลดเป็น 16 อัตโนมัติ

In [ ]:
CONVNEXT_NAME = run_model("convnextv2_tiny")
plot_curves(CONVNEXT_NAME)

## 7. โมเดล 3: DINOv3 ViT-S/16 [R5]

Vision Transformer ขนาดเล็ก (ประมาณ 21M พารามิเตอร์, patch 16×16) ที่ Meta pretrain แบบ self-supervised บนภาพจำนวนมาก ฟีเจอร์ที่ได้จับพื้นผิวและรายละเอียดเชิงพื้นที่ได้ดี จึงน่าจะเหมาะกับการแยกผ้าที่ลายคล้ายกัน (เช่น Polyester กับ Viscose) หัวจำแนกใช้ CLS token ต่อกับค่าเฉลี่ยของ patch tokens

ใช้ผ่าน `transformers` (`facebook/dinov3-vits16-pretrain-lvd1689m`) เทรนด้วย lr backbone 2e-5, lr หัวจำแนก 5e-4, weight decay 0.05 และเทรนเฉพาะหัวจำแนก 1 epoch แรก

**ต้องล็อกอิน Hugging Face และกดยอมรับเงื่อนไขของ DINOv3 ก่อน** (ดูส่วนที่ 4) ถ้าเข้าไม่ได้จะใช้ DINOv2 ViT-S/14 แทนอัตโนมัติ และชื่อในผลลัพธ์จะบอกชัดเจนว่าเป็นโมเดลสำรอง

In [ ]:
DINO_NAME = run_model("dinov3_vits16")
plot_curves(DINO_NAME)

## 8. เปรียบเทียบและเลือกโมเดล

**เกณฑ์ (กำหนดไว้ล่วงหน้าใน PRD)**
1. Macro-F1 บนชุด **Validation** สูงสุด (ถ้าเทรนหลาย seed ใช้ค่าเฉลี่ยของทุก seed)
2. ถ้าต่างกันไม่เกิน 0.01 เลือกตัวที่ inference เร็วกว่า
3. หลังเลือกแล้วจึงประเมินบนชุด **Test** ครั้งเดียว (Test ไม่ถูกใช้ในการจูนหรือเลือกโมเดล)

เวลา inference วัดแบบ end-to-end ต่อ 1 รูป (อ่านไฟล์ + preprocessing + ทำนาย) ด้วย predictor ตัวเดียวกับที่แอปใช้ โดยใช้ค่ามัธยฐานของ 50 รูปจากชุด Val ถ้าเทรนหลาย seed โมเดลแต่ละตัวจะแทนด้วย seed ที่ Val Macro-F1 สูงสุด

In [ ]:
MODEL_NAMES = list(NAMES)
assert len(MODEL_NAMES) == 3, f"ต้องมีโมเดล 3 ตัว แต่มี {MODEL_NAMES}"
PREDICTORS = {name: fl.TorchPredictor(RUNS[(KEY_OF[name], BEST_SEED[name])]["model"], DEVICE, IMG_SIZE)
              for name in MODEL_NAMES}


def time_inference(predictor, n=50):
    sample = [DATA_DIR / p for p in split_df["path"][masks["val"]].iloc[:n]]
    predictor.predict_proba(sample[:2])  # warm-up
    times = []
    for p in sample:
        t0 = time.perf_counter()
        predictor.predict_proba([p])
        times.append(time.perf_counter() - t0)
    return 1000 * float(np.median(times))


INFER_MS = {name: time_inference(PREDICTORS[name]) for name in tqdm(MODEL_NAMES, desc="timing")}
N_PARAMS = {name: sum(p.numel() for p in RUNS[(KEY_OF[name], BEST_SEED[name])]["model"].parameters()) / 1e6
            for name in MODEL_NAMES}

val_table = pd.DataFrame([{"model": name, **{f"val_{k}": v for k, v in evaluate(y_va, VAL_PROBA[name]).items()
                                             if k in ("accuracy", "macro_precision", "macro_recall", "macro_f1", "min_recall")},
                           "val_f1_mean": float(np.mean(SEED_F1[name])), "val_f1_std": float(np.std(SEED_F1[name])),
                           "n_seeds": len(SEEDS), "params_M": N_PARAMS[name],
                           "train_time_s": TRAIN_TIME[name], "infer_ms_per_img": INFER_MS[name]}
                          for name in MODEL_NAMES]).set_index("model")
display(val_table.round(3))

top_f1 = val_table["val_f1_mean"].max()
close = val_table[val_table["val_f1_mean"] >= top_f1 - 0.01]
BEST = close["infer_ms_per_img"].idxmin()
selection = {"best_model": BEST, "best_seed": BEST_SEED[BEST],
             "rule": "max mean Val Macro-F1 over seeds; ties within 0.01 -> faster inference",
             "candidates_within_0.01": close.index.tolist(), "seeds": list(SEEDS),
             "val_macro_f1_mean": val_table["val_f1_mean"].to_dict()}
(RESULTS_DIR / "selection.json").write_text(json.dumps(selection, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"โมเดลที่ดีที่สุด (เลือกจาก Val): {BEST}")

### 8.1 ประเมินบนชุด Test (ครั้งเดียว หลังเลือกโมเดลแล้ว)

การเลือกโมเดลถูกล็อกและบันทึกไว้ใน `results/selection.json` แล้ว ด้านล่างรายงานผล Test ของทั้ง 3 โมเดลเพื่อใช้ในรายงาน แต่ผลนี้ไม่ได้ย้อนกลับไปเปลี่ยนการเลือก

In [ ]:
TEST_PROBA = {}
for name in MODEL_NAMES:
    run = RUNS[(KEY_OF[name], BEST_SEED[name])]
    TEST_PROBA[name], _ = predict_loader(run["model"], make_loader("test", False, run["ckpt"]["batch_size"], BEST_SEED[name]))
TEST_METRICS = {name: evaluate(y_te, TEST_PROBA[name]) for name in MODEL_NAMES}

results = val_table.join(pd.DataFrame(TEST_METRICS).T.add_prefix("test_"))
results.insert(0, "selected", results.index == BEST)
results.to_csv(RESULTS_DIR / "results.csv")
display(results[["selected", "val_macro_f1", "test_accuracy", "test_macro_precision", "test_macro_recall",
                 "test_macro_f1", "test_min_recall", "params_M", "train_time_s", "infer_ms_per_img"]].round(3))

per_class = pd.DataFrame([{"model": name, "class": c, "precision": TEST_METRICS[name][f"precision_{c}"],
                           "recall": TEST_METRICS[name][f"recall_{c}"], "f1": TEST_METRICS[name][f"f1_{c}"]}
                          for name in MODEL_NAMES for c in CLASSES])
per_class.to_csv(RESULTS_DIR / "per_class_test.csv", index=False)
display(per_class.pivot(index="class", columns="model", values=["recall", "f1"]).round(3))

if len(SEEDS) > 1:  # ความแปรปรวนข้าม seed (รายงานเท่านั้น ไม่ใช้เลือกโมเดล)
    rows = []
    for name in MODEL_NAMES:
        f1s = []
        for s in SEEDS:
            run = RUNS[(KEY_OF[name], s)]
            p, _ = predict_loader(run["model"], make_loader("test", False, run["ckpt"]["batch_size"], s))
            f1s.append(evaluate(y_te, p)["macro_f1"])
        rows.append({"model": name, "test_f1_mean": np.mean(f1s), "test_f1_std": np.std(f1s),
                     **{f"seed_{s}": f for s, f in zip(SEEDS, f1s)}})
    seed_table = pd.DataFrame(rows).set_index("model")
    seed_table.to_csv(RESULTS_DIR / "test_f1_by_seed.csv")
    display(seed_table.round(3))

### 8.2 ช่วงความเชื่อมั่นและการทดสอบนัยสำคัญ

ชุด Test มีราว 330 รูป ผลที่ต่างกันเล็กน้อยอาจเป็นเพราะความบังเอิญของชุดข้อมูล จึงรายงาน
- **ช่วงความเชื่อมั่น 95% ของ Macro-F1** ด้วย bootstrap 1,000 รอบ (สุ่มรูป Test ซ้ำแบบใส่คืน)
- **McNemar test** เทียบโมเดลที่เลือกกับอีก 2 ตัวบนรูปชุดเดียวกัน (ดูเฉพาะรูปที่สองโมเดลทายต่างกัน) ถ้า p < 0.05 จึงถือว่าต่างกันจริง

ถ้าโมเดลใหม่ไม่ได้ต่างจาก ResNet50 อย่างมีนัยสำคัญ ให้เลือกตัวที่เร็วกว่าหรือเล็กกว่าตามกฎของ PRD

In [ ]:
from scipy.stats import binomtest
from sklearn.metrics import f1_score

rng = np.random.default_rng(SEED)
B = 1000
boot_idx = rng.integers(0, len(y_te), size=(B, len(y_te)))
best_ok = TEST_PROBA[BEST].argmax(1) == y_te
rows = []
for name in MODEL_NAMES:
    pred = TEST_PROBA[name].argmax(1)
    f1s = [f1_score(y_te[i], pred[i], labels=range(len(CLASSES)), average="macro", zero_division=0) for i in boot_idx]
    lo, hi = np.percentile(f1s, [2.5, 97.5])
    row = {"model": name, "test_macro_f1": TEST_METRICS[name]["macro_f1"], "ci95_low": lo, "ci95_high": hi}
    if name != BEST:
        ok = pred == y_te
        b, c = int((best_ok & ~ok).sum()), int((~best_ok & ok).sum())
        row.update({"best_right_other_wrong": b, "other_right_best_wrong": c,
                    "mcnemar_p": binomtest(b, b + c, 0.5).pvalue if b + c else 1.0})
    rows.append(row)
sig = pd.DataFrame(rows).set_index("model")
sig.to_csv(RESULTS_DIR / "significance_test.csv")
display(sig.round(4))
for name, p in sig["mcnemar_p"].dropna().items():
    verdict = "ต่างกันอย่างมีนัยสำคัญ" if p < 0.05 else "ไม่ต่างกันอย่างมีนัยสำคัญ"
    print(f"{BEST} เทียบกับ {name}: p = {p:.3f} -> {verdict} (ที่ระดับ 0.05)")

In [ ]:
# ผลแยกตามแหล่งข้อมูล: ตรวจว่าโมเดลเรียนรู้ "ชนิดผ้า" ไม่ใช่ "แหล่งข้อมูล"
src_te = split_df["source"][masks["test"]].to_numpy()
per_source = pd.DataFrame([{"model": name, "source": s, "n": int((src_te == s).sum()),
                            **{k: v for k, v in evaluate(y_te[src_te == s], TEST_PROBA[name][src_te == s]).items()
                               if k in ("accuracy", "macro_f1", "min_recall")}}
                           for name in MODEL_NAMES for s in SOURCES if (src_te == s).any()])
per_source.to_csv(RESULTS_DIR / "per_source_test.csv", index=False)
display(per_source.round(3))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))
for ax, name in zip(axes, MODEL_NAMES):
    cm = confusion_matrix(y_te, TEST_PROBA[name].argmax(1), labels=range(len(CLASSES)))
    cm_norm = cm / cm.sum(1, keepdims=True).clip(min=1)
    annot = np.array([[f"{c}\n{r:.0%}" for c, r in zip(row, rrow)] for row, rrow in zip(cm, cm_norm)])
    sns.heatmap(cm_norm, annot=annot, fmt="", cmap="Blues", vmin=0, vmax=1, cbar=False,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set(title=f"{name}{' (selected)' if name == BEST else ''}  F1={TEST_METRICS[name]['macro_f1']:.3f}",
           xlabel="Predicted", ylabel="True")
savefig(fig, "confusion_matrices_test.png")
plt.show()

In [ ]:
# เทียบกับเป้าหมายใน PRD (โมเดลที่เลือก)
tm = TEST_METRICS[BEST]
targets = pd.DataFrame([
    ("Macro-F1 (Test)", "≥ 0.85", tm["macro_f1"], tm["macro_f1"] >= 0.85),
    ("Accuracy (Test)", "≥ 85%", tm["accuracy"], tm["accuracy"] >= 0.85),
    ("Recall ต่ำสุดรายคลาส (Test)", "≥ 0.75", tm["min_recall"], tm["min_recall"] >= 0.75),
    ("Inference ต่อรูป (ms)", "< 1000", INFER_MS[BEST], INFER_MS[BEST] < 1000),
], columns=["ตัวชี้วัด", "เป้า", "ผล", "ผ่าน"]).set_index("ตัวชี้วัด")
display(targets.round(3))

## 9. Error analysis

แสดงรูปใน Test ที่โมเดลที่เลือกทายผิดโดยมั่นใจสูงสุด 12 รูป และคู่คลาสที่สับสนบ่อย ใช้ประกอบการอภิปรายในรายงาน (R3 พบว่าคู่ Cotton–Polyester และ Satin–Silk สับสนกันบ่อยเพราะลายผิวคล้ายกัน)

In [ ]:
test_df = split_df[masks["test"]].reset_index(drop=True)
proba = TEST_PROBA[BEST]
pred, conf = proba.argmax(1), proba.max(1)
wrong = np.flatnonzero(pred != y_te)
top_wrong = wrong[np.argsort(-conf[wrong])][:12]
print(f"{BEST}: ทายผิด {len(wrong)} จาก {len(y_te)} รูปใน Test")

if len(top_wrong):
    ncol = 4
    nrow = int(np.ceil(len(top_wrong) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(13, 3.4 * nrow), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, i in zip(axes.ravel(), top_wrong):
        img = fl.load_pil(DATA_DIR / test_df.path[i])
        img.thumbnail((256, 256))
        ax.imshow(img)
        ax.set_title(f"true {CLASSES[y_te[i]]} -> {CLASSES[pred[i]]}\nconf {conf[i]:.0%} | {test_df.source[i]}", fontsize=9)
    fig.suptitle(f"{BEST}: most confident mistakes (test)")
    savefig(fig, "error_top_confident_mistakes.png")
    plt.show()

cm = confusion_matrix(y_te, pred, labels=range(len(CLASSES)))
pairs = pd.DataFrame([{"true": CLASSES[i], "predicted": CLASSES[j], "count": int(cm[i, j]),
                       "% of true class": cm[i, j] / max(cm[i].sum(), 1)}
                      for i in range(len(CLASSES)) for j in range(len(CLASSES)) if i != j and cm[i, j] > 0])
if len(pairs):
    display(pairs.sort_values("count", ascending=False).head(6).style.format({"% of true class": "{:.1%}"}))

## 10. Export

บันทึกโมเดลที่ดีที่สุดเป็น `models/best_model.pth` พร้อม `models/model_meta.json` ที่ระบุชื่อโมเดล คลาส preprocessing (รวม config ของโมเดลตระกูล Hugging Face เพื่อให้แอปสร้างโครงโมเดลได้โดยไม่ต้องใช้อินเทอร์เน็ต) และผล Test แอปอ่านข้อมูลจากไฟล์นี้ แล้วตรวจว่าโมเดลที่โหลดผ่าน `fl.load_predictor` (ทางเดียวกับแอป) ให้ผลตรงกับใน notebook

In [ ]:
for f in ("best_model.pkl", "best_model.pth"):
    (MODELS_DIR / f).unlink(missing_ok=True)

best_run = RUNS[(KEY_OF[BEST], BEST_SEED[BEST])]
best_ckpt = best_run["ckpt"]
model_file, model_type = "best_model.pth", "torch"
torch.save({k: v.detach().cpu().contiguous() for k, v in best_run["model"].state_dict().items()}, MODELS_DIR / model_file)
preprocessing = {"arch": KEY_OF[BEST], "img_size": IMG_SIZE, "resize": int(round(IMG_SIZE * 256 / 224)),
                 "mean": list(fl.IMAGENET_MEAN), "std": list(fl.IMAGENET_STD),
                 "hf_config": best_ckpt.get("hf_config"), "hf_repo": best_ckpt.get("hf_repo"),
                 "note": "RGB, EXIF-rotated, resize shorter side, center crop, ImageNet normalize"}
hyperparams = {**best_ckpt["hparams"], "batch_size": best_ckpt["batch_size"], "epochs_trained": len(best_ckpt["history"])}

meta = {"model_name": f"{BEST} (pretrained, fine-tuned)",
        "model_key": BEST, "model_type": model_type, "model_file": model_file, "classes": CLASSES,
        "preprocessing": preprocessing, "hyperparameters": {k: str(v) if not isinstance(v, (int, float)) else v
                                                            for k, v in hyperparams.items()},
        "conf_threshold": CONF_THRESHOLD, "test_metrics": TEST_METRICS[BEST],
        "val_metrics": evaluate(y_va, VAL_PROBA[BEST]), "infer_ms_per_img": INFER_MS[BEST],
        "n_images": {s: int(masks[s].sum()) for s in masks}, "seed": BEST_SEED[BEST], "data_fingerprint": FINGERPRINT,
        "exported_at": time.strftime("%Y-%m-%d %H:%M:%S")}
(MODELS_DIR / "model_meta.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

# ตรวจว่าโหลดแบบเดียวกับแอปแล้วได้ผลตรงกับ notebook
app_pred = fl.load_predictor(MODELS_DIR)
chk = [DATA_DIR / p for p in test_df.path[:16]]
p_app = app_pred.predict_proba(chk)
diff = np.abs(p_app - TEST_PROBA[BEST][:16]).max()
assert (p_app.argmax(1) == TEST_PROBA[BEST][:16].argmax(1)).all() and diff < 0.02, f"ผลไม่ตรง (max diff {diff:.4f})"
print(f"Export {BEST} -> {MODELS_DIR / model_file} ; ผลจาก load_predictor ตรงกับ notebook (max diff {diff:.1e})")
del app_pred

## 11. เขียนแอป Streamlit (`app.py`)

แอปหน้าเดียว: อัปโหลดรูป (jpg/png/webp, หลายรูปได้) หรือถ่ายผ่านกล้อง แล้วแสดงชนิดผ้า ความมั่นใจของทั้ง 4 คลาส คำเตือนเมื่อความมั่นใจ < 60% และคำแนะนำการรีไซเคิล

| ID | ความต้องการ | ที่ทำไว้ |
|---|---|---|
| A1 | โหลดโมเดลครั้งเดียว | `st.cache_resource` |
| A2 | preprocessing ตรงกับตอนเทรน | `fl.load_predictor` อ่าน `model_meta.json` และใช้โค้ดเดียวกับ notebook |
| A3 | คลาส + bar chart 4 คลาส | ✓ |
| A4 | เตือนเมื่อความมั่นใจ < 60% | ✓ |
| A5 | ไฟล์ผิดประเภท/เสีย แสดงข้อความภาษาไทย ไม่มี traceback | ✓ |
| A6 | Sidebar: ชื่อโมเดล, ผล Test, ข้อจำกัด | ✓ |
| A7 | หลายรูป → ตารางสรุป + ดาวน์โหลด CSV | ✓ |
| A8 | คำแนะนำการรีไซเคิล | ✓ |
| A9 | Test-time augmentation (flip) | สวิตช์ใน sidebar |

รูปที่อัปโหลดประมวลผลในหน่วยความจำเท่านั้น ไม่บันทึกลงดิสก์และไม่ส่งออกนอกเครื่อง

In [ ]:
%%writefile app.py
"""FabricSort: แอป Streamlit จำแนกชนิดเศษผ้าเพื่อนำไปรีไซเคิล

สร้างจาก fabric_classification.ipynb (ส่วนที่ 11) รันด้วย: streamlit run app.py
"""
import io
from pathlib import Path

import numpy as np
import pandas as pd
import streamlit as st
from PIL import Image, ImageOps

import fabric_lib as fl

MODEL_DIR = Path(__file__).resolve().parent / "models"

TH_NAME = {"Cotton": "ผ้าฝ้าย", "Polyester": "โพลีเอสเตอร์", "Silk": "ผ้าไหม", "Viscose": "วิสคอส (เรยอน)"}
ADVICE = {
    "Cotton": "เส้นใยธรรมชาติ (เซลลูโลส) รีไซเคิลเชิงกลได้ดี: ตัด/ฉีกเป็นเส้นใยแล้วปั่นเป็นด้ายใหม่ "
              "หรือทำเป็นวัสดุบุ ฉนวน และผ้าเช็ด ควรแยกตามสีเพื่อลดการฟอกย้อมซ้ำ",
    "Polyester": "พลาสติกชนิด PET: รีไซเคิลเชิงกลโดยหลอมเป็นเม็ดแล้วรีดเป็นเส้นใยใหม่ "
                 "หรือรีไซเคิลเชิงเคมี (depolymerization) ย่อยกลับเป็นโมโนเมอร์เพื่อผลิตโพลีเอสเตอร์คุณภาพเท่าเดิม",
    "Silk": "เส้นใยโปรตีนมูลค่าสูง: เหมาะกับการนำกลับมาใช้ใหม่หรือ upcycling "
            "หรือรีไซเคิลเชิงกลเป็นเส้นใยสั้นสำหรับผ้าผสม ไม่ควรปนกับผ้าใยสังเคราะห์",
    "Viscose": "เซลลูโลสที่ผ่านกระบวนการทางเคมี (regenerated cellulose): เส้นใยสั้นและอ่อนลงเมื่อรีไซเคิลเชิงกล "
               "จึงเหมาะกับรีไซเคิลเชิงเคมี คือละลายเป็นเยื่อเซลลูโลสแล้วผลิตเส้นใยใหม่ (เช่น lyocell)",
}

st.set_page_config(page_title="FabricSort", page_icon="🧵", layout="wide")


@st.cache_resource(show_spinner="กำลังโหลดโมเดล...")
def get_predictor(meta_mtime):
    # meta_mtime เป็น key ของ cache: เมื่อ notebook export โมเดลใหม่ แอปจะโหลดใหม่เองโดยไม่ต้องปิดเปิด
    return fl.load_predictor(MODEL_DIR)


def read_image(uploaded):
    """คืน PIL RGB หรือ None ถ้าไฟล์ไม่ใช่รูปหรือไฟล์เสีย (อ่านในหน่วยความจำ ไม่บันทึกลงดิสก์)"""
    try:
        img = Image.open(io.BytesIO(uploaded.getvalue()))
        img.load()
        return ImageOps.exif_transpose(img).convert("RGB")
    except Exception:
        return None


def show_result(img, name, p, classes, threshold):
    k = int(np.argmax(p))
    cls, conf = classes[k], float(p[k])
    col_img, col_res = st.columns([1, 1.3])
    with col_img:
        st.image(img, caption=name)
    with col_res:
        st.subheader(f"{cls} · {TH_NAME.get(cls, cls)}")
        st.metric("ความมั่นใจ", f"{conf:.1%}")
        if conf < threshold:
            st.warning("⚠️ ไม่แน่ใจ ควรให้คนตรวจซ้ำ")
        chart = pd.DataFrame({"ความมั่นใจ (%)": np.round(np.asarray(p) * 100, 1)},
                             index=pd.Index(classes, name="ชนิดผ้า"))
        st.bar_chart(chart)
        st.info(f"♻️ **การรีไซเคิล:** {ADVICE.get(cls, '-')}")


try:
    predictor = get_predictor((MODEL_DIR / "model_meta.json").stat().st_mtime)
except FileNotFoundError:
    st.error("ไม่พบไฟล์โมเดลในโฟลเดอร์ models/ กรุณารัน fabric_classification.ipynb ให้ครบก่อน")
    st.stop()
except Exception:
    st.error("โหลดโมเดลไม่สำเร็จ กรุณารัน notebook ส่วนที่ 10 (Export) ใหม่อีกครั้ง")
    st.stop()

meta = predictor.meta
classes = meta["classes"]
threshold = float(meta.get("conf_threshold", 0.6))

with st.sidebar:
    st.header("ข้อมูลโมเดล")
    st.markdown(f"**โมเดล:** {meta['model_name']}")
    tm = meta["test_metrics"]
    c1, c2 = st.columns(2)
    c1.metric("Accuracy (Test)", f"{tm['accuracy']:.1%}")
    c2.metric("Macro-F1 (Test)", f"{tm['macro_f1']:.3f}")
    st.caption(f"ทดสอบกับ {meta['n_images']['test']:,} รูป · ส่งออกเมื่อ {meta['exported_at']}")
    tta = st.toggle("Test-time augmentation (เฉลี่ยผลกับภาพกลับด้าน)", value=True)
    st.subheader("ข้อจำกัด")
    st.markdown(
        "- จำแนกได้เพียง 4 ชนิด: Cotton, Polyester, Silk, Viscose\n"
        "- ไม่รองรับผ้าผสม (blend) หรือผ้าชนิดอื่น ระบบจะตอบเป็น 1 ใน 4 ชนิดเสมอ\n"
        "- 1 รูปควรมีผ้าชิ้นเดียว ถ่ายใกล้และเต็มเฟรม\n"
        "- เทรนจากชุดข้อมูลสาธารณะ ความแม่นยำกับรูปจากโรงงานจริงอาจต่ำกว่านี้\n"
        f"- ความมั่นใจต่ำกว่า {threshold:.0%} ควรให้คนตรวจซ้ำ\n"
        "- รูปประมวลผลในเครื่องนี้เท่านั้น ไม่ถูกบันทึกหรือส่งออกไปที่อื่น"
    )

st.title("🧵 FabricSort")
st.markdown("ระบบจำแนกเศษผ้าเพื่อนำไปรีไซเคิล: อัปโหลดหรือถ่ายรูปเศษผ้า แล้วระบบจะบอกว่าเป็น "
            "**Cotton, Polyester, Silk หรือ Viscose** พร้อมค่าความมั่นใจ")
with st.expander("📸 คำแนะนำการถ่ายรูป", expanded=False):
    st.markdown("1. ถ่ายใกล้ ๆ ให้ผ้าเต็มเฟรม ไม่มีพื้นหลังหรือวัตถุอื่น\n"
                "2. ใช้แสงสม่ำเสมอ หลีกเลี่ยงเงาและแสงสะท้อนจ้า\n"
                "3. ถือกล้องให้นิ่ง ให้เห็นลายเส้นใยชัดเจน\n"
                "4. 1 รูปต่อผ้า 1 ชิ้น")

tab_up, tab_cam = st.tabs(["📁 อัปโหลดรูป", "📷 ถ่ายรูป"])
with tab_up:
    files = st.file_uploader("เลือกรูปผ้า (jpg, png, webp) เลือกได้หลายรูป",
                             type=["jpg", "jpeg", "png", "webp"], accept_multiple_files=True)
with tab_cam:
    shot = st.camera_input("ถ่ายรูปผ้า")

inputs = list(files or []) + ([shot] if shot is not None else [])
if not inputs:
    st.info("อัปโหลดหรือถ่ายรูปผ้าเพื่อเริ่มจำแนก")
    st.stop()

images, names = [], []
for f in inputs:
    img = read_image(f)
    if img is None:
        st.error(f"ไฟล์ “{f.name}” ไม่ใช่รูปภาพที่รองรับหรือไฟล์เสีย กรุณาใช้ไฟล์ jpg, png หรือ webp")
    else:
        images.append(img)
        names.append(f.name)
if not images:
    st.stop()

try:
    with st.spinner("กำลังจำแนก..."):
        probs = predictor.predict_proba(images, tta=tta)
except Exception:
    st.error("เกิดข้อผิดพลาดระหว่างจำแนกรูป กรุณาลองใหม่อีกครั้งหรือใช้รูปอื่น")
    st.stop()

if len(images) == 1:
    show_result(images[0], names[0], probs[0], classes, threshold)
else:
    rows = []
    for name, p in zip(names, probs):
        k = int(np.argmax(p))
        rows.append({"ไฟล์": name, "ชนิดผ้า": classes[k], "ความมั่นใจ (%)": round(float(p[k]) * 100, 1),
                     "ควรตรวจซ้ำ": "ใช่" if p[k] < threshold else "",
                     **{f"{c} (%)": round(float(v) * 100, 1) for c, v in zip(classes, p)}})
    table = pd.DataFrame(rows)
    st.subheader(f"สรุปผล {len(images)} รูป")
    st.dataframe(table, hide_index=True)
    st.download_button("⬇️ ดาวน์โหลดผลเป็น CSV", table.to_csv(index=False).encode("utf-8-sig"),
                       file_name="fabricsort_results.csv", mime="text/csv")
    for img, name, p in zip(images, names, probs):
        with st.expander(f"{name}: {classes[int(np.argmax(p))]} ({float(np.max(p)):.0%})"):
            show_result(img, name, p, classes, threshold)

## 12. เปิดแอป

รัน Streamlit แบบ background แล้วเปิดเบราว์เซอร์ที่ http://localhost:8501 log อยู่ที่ `results/streamlit.log` ถ้ารัน cell นี้ซ้ำ แอปเดิมจะถูกปิดแล้วเปิดใหม่ (เพื่อโหลดโมเดลล่าสุด)

ปิดแอปเมื่อใช้เสร็จ: รัน `app_proc.terminate()` หรือ restart kernel หรือรันแอปเองนอก notebook ด้วย `streamlit run app.py`

In [ ]:
import socket, subprocess, webbrowser

APP_URL = f"http://localhost:{APP_PORT}"


def port_in_use(port):
    with socket.socket() as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", port)) == 0


if not LAUNCH_APP:
    print("LAUNCH_APP = False ข้ามการเปิดแอป (รันเองได้ด้วย: streamlit run app.py)")
else:
    if "app_proc" in globals() and app_proc.poll() is None:
        app_proc.terminate()
        app_proc.wait(timeout=10)
        time.sleep(1)
    if port_in_use(APP_PORT):
        print(f"พอร์ต {APP_PORT} ถูกใช้อยู่ (อาจเป็นแอปเดิมที่เปิดจากที่อื่น) ปิดโปรแกรมนั้นก่อน หรือเปลี่ยน APP_PORT ใน Config")
    else:
        app_log = open(RESULTS_DIR / "streamlit.log", "w", encoding="utf-8")
        app_proc = subprocess.Popen(
            [sys.executable, "-m", "streamlit", "run", str(PROJECT_DIR / "app.py"),
             "--server.address", "localhost", "--server.port", str(APP_PORT), "--server.headless", "true",
             "--browser.gatherUsageStats", "false"],
            cwd=PROJECT_DIR, stdout=app_log, stderr=subprocess.STDOUT)
        for _ in range(120):
            if port_in_use(APP_PORT) or app_proc.poll() is not None:
                break
            time.sleep(0.5)
        if port_in_use(APP_PORT):
            display(HTML(f'<b>FabricSort พร้อมใช้งาน:</b> <a href="{APP_URL}" target="_blank">{APP_URL}</a>'))
            webbrowser.open(APP_URL)
        else:
            print("เปิดแอปไม่สำเร็จ ดู log ที่", RESULTS_DIR / "streamlit.log")